In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Nuclear_fusion.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/List_of_common_misconceptions_about_science__technology__and_mathematics.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Existence.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Scientific_theory.txt


In [14]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_KEY = user_secrets.get_secret("wandb-key")


In [15]:
import wandb 
wandb.login(key=WB_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [16]:
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
KAGGLE_CORPUS_PATH = "/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus"
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
LABELS = OPTION_COLS
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

In [17]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()


def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual in predicted:
        return 1 / (predicted.index(actual) + 1)
    return 0


def mapk(actuals, predictions, k=3):
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, predictions)]))


In [18]:
import re
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
train_df.head()
train_df.info()


train_df.fillna("None", inplace=True)


for col in OPTION_COLS + ['prompt']:
    if col in train_df.columns:
        train_df[col] = train_df[col].apply(clean_text)
print("Data cleaned")


def tokenize_text(text):
    return word_tokenize(text)


for col in OPTION_COLS + ['prompt']:
    train_df[f'{col}_tokens'] = train_df[col].apply(tokenize_text)
print("Tokenization done. Example:", train_df['prompt_tokens'].iloc[0][:10])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB
Data cleaned
Tokenization done. Example: ['pick', 'the', 'best', 'possible', 'answer', ':', 'what', 'is', 'martin', 'heidegger']


## Baseline TFIDF + Cosing Similarity 

In [19]:
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_predictions, actuals = [], []

for idx, row in train_df.iterrows():
    corpus = [row['prompt']] + [row[c] for c in OPTION_COLS]
    tfidf_matrix = vectorizer.fit_transform(corpus)
    prompt_vector = tfidf_matrix[0:1]
    options_matrix = tfidf_matrix[1:]

    similarities = cosine_similarity(prompt_vector, options_matrix).flatten()
    top_3_idx = similarities.argsort()[-3:][::-1]
    tfidf_predictions.append([LABELS[i] for i in top_3_idx])

    if 'answer' in train_df.columns:
        actuals.append(row['answer'])

if actuals:
    tfidf_map3 = mapk(actuals, tfidf_predictions, k=3)
    print(f'Baseline TF-IDF MAP@3: {tfidf_map3:.4f}')
else:
    print('No answer column found')

Baseline TF-IDF MAP@3: 0.3260


## Baseline 2 Word2Vec + cosine similarity 

In [20]:
from gensim.models import Word2Vec

all_tokenized_sentences = []
for col in OPTION_COLS + ['prompt']:
    all_tokenized_sentences.extend(train_df[f'{col}_tokens'].tolist())

w2v_model = Word2Vec(
    sentences=all_tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    seed=42,
)


def avg_word2vec_vector(tokens, model):
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)


w2v_predictions = []
for idx, row in train_df.iterrows():
    prompt_vec = avg_word2vec_vector(row['prompt_tokens'], w2v_model).reshape(1, -1)
    option_vecs = np.stack([
        avg_word2vec_vector(row[f'{c}_tokens'], w2v_model) for c in OPTION_COLS
    ])
    sims = cosine_similarity(prompt_vec, option_vecs).flatten()
    top_3_idx = sims.argsort()[-3:][::-1]
    w2v_predictions.append([LABELS[i] for i in top_3_idx])

if actuals:
    w2v_map3 = mapk(actuals, w2v_predictions, k=3)
    print(f'Baseline Word2Vec MAP@3: {w2v_map3:.4f}')
    print(f'TF-IDF vs Word2Vec MAP@3: {tfidf_map3:.4f} vs {w2v_map3:.4f}')

Baseline Word2Vec MAP@3: 0.3184
TF-IDF vs Word2Vec MAP@3: 0.3260 vs 0.3184


# Milestone 2 

In [21]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM, pipeline
from sentence_transformers import SentenceTransformer, util

device = "cuda" if torch.cuda.is_available() else "cpu"

df_pandas = pd.read_csv(f'{DATA_DIR}/train.csv').fillna('None')
dataset = Dataset.from_pandas(df_pandas)


def combine_text_fn(example):
    return {"combined_text": f"{example['prompt']} {example['A']}"}


dataset = dataset.map(combine_text_fn)

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')


def tokenize_prompts(examples):
    return tokenizer(examples['prompt'], padding='max_length', truncation=True, max_length=128)


tokenized_dataset = dataset.map(tokenize_prompts, batched=True)
input_ids_shape = np.array(tokenized_dataset['input_ids']).shape

bert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)
row_0_prompt = dataset[0]['prompt']
inputs_0 = tokenizer(row_0_prompt, return_tensors='pt').to(device)

with torch.no_grad():
    outputs_0 = bert_model(**inputs_0)

last_hidden_state = outputs_0.last_hidden_state
cls_vector = last_hidden_state[0, 0, :5].tolist()

model_att = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True).to(device)
text_att = 'Light-ion fusion is a technique.'
inputs_att = tokenizer(text_att, return_tensors='pt').to(device)
tokens_att = tokenizer.convert_ids_to_tokens(inputs_att['input_ids'][0])
fusion_idx = tokens_att.index('fusion')

with torch.no_grad():
    outputs_att = model_att(**inputs_att)

attention_matrix = outputs_att.attentions[-1][0, 0]
weight_cls_to_fusion = attention_matrix[0, fusion_idx].item()

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
emb_prompt = st_model.encode(dataset[0]['prompt'], convert_to_tensor=True)
emb_opt_b = st_model.encode(dataset[0]['B'], convert_to_tensor=True)
sim_score = util.cos_sim(emb_prompt, emb_opt_b).item()

all_prompts = df_pandas['prompt'].tolist()
opt_cols = [df_pandas[c].tolist() for c in LABELS]
actual_answers = df_pandas['answer'].tolist() if 'answer' in df_pandas.columns else []

prompt_embs = st_model.encode(all_prompts, convert_to_tensor=True)
opt_embs = [st_model.encode(col, convert_to_tensor=True) for col in opt_cols]

tf_idf_vectorizer = TfidfVectorizer(stop_words='english')
minilm_preds, tfidf_preds = [], []

for idx, row in df_pandas.iterrows():
    corpus = [row['prompt']] + [row[l] for l in LABELS]
    try:
        tfidf_mat = tf_idf_vectorizer.fit_transform(corpus)
        sims_tfidf = cosine_similarity(tfidf_mat[0:1], tfidf_mat[1:]).flatten()
        top_3_tfidf = [LABELS[i] for i in sims_tfidf.argsort()[-3:][::-1]]
    except Exception:
        top_3_tfidf = ['A', 'B', 'C']
    tfidf_preds.append(top_3_tfidf)

    p_emb = prompt_embs[idx]
    sims_minilm = [util.cos_sim(p_emb, opt_embs[o_idx][idx]).item() for o_idx in range(5)]
    top_3_minilm = [LABELS[i] for i in np.argsort(sims_minilm)[-3:][::-1]]
    minilm_preds.append(top_3_minilm)

map3_minilm = mapk(actual_answers, minilm_preds, k=3) if actual_answers else None

improvement_count = 0
for a, t_pred, m_pred in zip(actual_answers, tfidf_preds, minilm_preds):
    if (a in m_pred) and (a not in t_pred):
        improvement_count += 1

classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=0 if device == "cuda" else -1,
)
prompt_idx_1 = dataset[1]['prompt']
candidates = [dataset[1]['A'], dataset[1]['B'], dataset[1]['C']]

res_softmax = classifier(prompt_idx_1, candidate_labels=candidates, multi_label=False)
top_prob_softmax = res_softmax['scores'][0]

res_sigmoid = classifier(prompt_idx_1, candidate_labels=candidates, multi_label=True)
abs_diff = abs(sum(res_softmax['scores']) - sum(res_sigmoid['scores']))

slm_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
slm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(device)


def ask_slm(prompt_text, max_new_tokens=5):
    inputs = slm_tokenizer(prompt_text, return_tensors="pt").to(device)
    outputs = slm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return slm_tokenizer.decode(outputs[0], skip_special_tokens=True)


prompt_slm = (
    f"Question: {dataset[0]['prompt']}. Is the correct answer A: {dataset[0]['A']} "
    f"or B: {dataset[0]['B']}? Answer with just the letter A or B."
)
slm_out = ask_slm(prompt_slm)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [22]:
print(f"Q1: Character length at index 51: {len(dataset[51]['combined_text'])}")
print(f"Q2: Vocabulary Size: {tokenizer.vocab_size}")
print(f"Q3: [SEP] Token ID: {tokenizer.convert_tokens_to_ids('[SEP]')}")
print(f"Q4: Geometric shape of input_ids tensor: {input_ids_shape}")
print(f"Q5: Dimensionality of each individual attention head: {768 // 12}")
print(f"Q6: Shape of last_hidden_state tensor: {list(last_hidden_state.shape)}")
print(f"Q7: Sum of first 5 float values in [CLS] vector: {round(sum(cls_vector), 4)}")
print(f"Q8: Attention weight from [CLS] to 'fusion': {round(weight_cls_to_fusion, 4)}")
print(f"Q9: Cosine similarity between prompt and Option B: {round(sim_score, 4)}")
if map3_minilm is not None:
    print(f"Q10 Part 1: Final MAP@3 score of MiniLM pipeline: {round(map3_minilm, 4)}")
    print(f"Q10 Part 2: Number of questions saved by MiniLM: {improvement_count}")

Q1: Character length at index 51: 614
Q2: Vocabulary Size: 30522
Q3: [SEP] Token ID: 102
Q4: Geometric shape of input_ids tensor: (2000, 128)
Q5: Dimensionality of each individual attention head: 64
Q6: Shape of last_hidden_state tensor: [1, 31, 768]
Q7: Sum of first 5 float values in [CLS] vector: -1.2001
Q8: Attention weight from [CLS] to 'fusion': 0.1025
Q9: Cosine similarity between prompt and Option B: 0.7658
Q10 Part 1: Final MAP@3 score of MiniLM pipeline: 0.4231
Q10 Part 2: Number of questions saved by MiniLM: 488


## Milestone 3 

In [23]:
!pip install -q langchain-text-splitters langchain-huggingface faiss-cpu langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-ser

In [24]:
import re

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').fillna('None')

print("=== DATASET DIAGNOSTICS ===")
print(f"Total Questions: {len(train_df)}")

print("\n--- Sample Prompts & Topics ---")
for idx, row in train_df.sample(5, random_state=42).iterrows():
    print(f"ID {row['id']}: {row['prompt'][:120]}...")

def extract_sample_entities(text):
   
    cleaned = re.sub(r'(?i)(pick the best|what is|which of the following|according to)', '', text)
    
    entities = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', cleaned)
    return entities[:3]

print("\n--- Extracted Sample Entities for Retrieval ---")
for idx, row in train_df.head(5).iterrows():
    entities = extract_sample_entities(row['prompt'])
    print(f"Prompt: {row['prompt'][:60]}... -> Entities: {entities}")

=== DATASET DIAGNOSTICS ===
Total Questions: 2000

--- Sample Prompts & Topics ---
ID 1861: Select the most accurate option: What is the propagation constant in sinusoidal waves? carefully....
ID 354: Pick the best possible answer: What is the most popular explanation for the shower-curtain effect? among the listed opti...
ID 1334: Pick the best possible answer: What is the Kelvin-Helmholtz instability and how does it affect Earth's magnetosphere? fr...
ID 906: Identify the correct statement: What is a crossover experiment? based on the given context....
ID 1290: Choose the correct answer: What is the relationship between interstellar and cometary chemistry? from the following choi...

--- Extracted Sample Entities for Retrieval ---
Prompt: Pick the best possible answer: What is Martin Heidegger's vi... -> Entities: ['Martin Heidegger']
Prompt: What is accelerator-based light-ion fusion?... -> Entities: []
Prompt: Determine the correct option: What is the term used in astro... -> Entit

In [25]:
import os
import glob
import re
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Set Exact Paths


train_df = pd.read_csv(f'{DATA_DIR}/train.csv').fillna('None')

def clean_corpus_text(text):
    text = text.strip()
    text = re.sub(r' {2,}', ' ', text)
    return text.replace("\r", "")

# 2. Load Documents
docs = []
if os.path.exists(KAGGLE_CORPUS_PATH):
    txt_files = glob.glob(os.path.join(KAGGLE_CORPUS_PATH, "*.txt"))
    for file_path in txt_files:
        with open(file_path, "r", encoding="utf-8") as f:
            text = clean_corpus_text(f.read())
            if text:
                docs.append(Document(page_content=text, metadata={"source": file_path}))

print(f"Successfully loaded {len(docs)} documents into memory.")

# 3. Chunking & FAISS Vector Database Setup
text_splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=80)
chunks = text_splitter.split_documents(docs)
print(f"Created {len(chunks)} text chunks.")

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},  # Change to 'cuda' if GPU is enabled on Kaggle
)

vector_store = FAISS.from_documents(chunks, embeddings_model)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
print("FAISS vector database built successfully!")

def build_rag_prompt(row, context_str):
    return f"""Context: {context_str}
Question: {row['prompt']}
Options:
A) {row['A']}
B) {row['B']}
C) {row['C']}
D) {row['D']}
E) {row['E']}
Based on the context above, answer with just the single best option letter (A, B, C, D, or E)."""

def build_baseline_prompt(row):
    return f"""Question: {row['prompt']}
Options:
A) {row['A']}
B) {row['B']}
C) {row['C']}
D) {row['D']}
E) {row['E']}
Answer with just the single best option letter (A, B, C, D, or E)."""

def extract_letter(generated_text):
    if generated_text:
        for ch in generated_text.strip().upper():
            if ch in ["A", "B", "C", "D", "E"]:
                return ch
    return None

SAMPLE_SIZE = min(50, len(train_df))
eval_df = train_df.head(SAMPLE_SIZE)

baseline_correct, rag_correct, evaluated = 0, 0, 0
augmented_prompts = []

for idx, row in eval_df.iterrows():
    retrieved_docs = retriever.invoke(row['prompt'])
    context_str = " ".join(doc.page_content for doc in retrieved_docs)

    baseline_prompt = build_baseline_prompt(row)
    rag_prompt = build_rag_prompt(row, context_str)
    augmented_prompts.append(rag_prompt)

    if 'answer' not in row or pd.isna(row.get('answer')):
        continue

    baseline_answer = extract_letter(ask_slm(baseline_prompt))
    rag_answer = extract_letter(ask_slm(rag_prompt))

    evaluated += 1
    if baseline_answer == row['answer']:
        baseline_correct += 1
    if rag_answer == row['answer']:
        rag_correct += 1

train_df['augmented_rag_prompt'] = augmented_prompts + [None] * (len(train_df) - len(augmented_prompts))
train_df.to_csv("train_augmented_rag.csv", index=False)

if evaluated:
    print(f"\nSample size evaluated: {evaluated}")
    print(f"Baseline accuracy: {baseline_correct / evaluated:.4f}")
    print(f"RAG accuracy:      {rag_correct / evaluated:.4f}")

/tmp/ipykernel_58/1544647011.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Successfully loaded 3 documents into memory.
Created 829 text chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector database built successfully!


Token indices sequence length is longer than the specified maximum sequence length for this model (626 > 512). Running this sequence through the model will result in indexing errors



Sample size evaluated: 50
Baseline accuracy: 0.2800
RAG accuracy:      0.2800


In [26]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch  
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset as TorchDataset
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb  # noqa: F811
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from peft import LoraConfig, get_peft_model, TaskType
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset as HFDataset

device = "cuda" if torch.cuda.is_available() else "cpu"

train_df = pd.read_csv(f'{DATA_DIR}/train.csv').fillna('None')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv').fillna('None')

for col in ['prompt'] + OPTION_COLS:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)


train_split_df, val_split_df = train_test_split(
    train_df, test_size=0.1, random_state=42, stratify=train_df['answer']
)
train_split_df = train_split_df.reset_index(drop=True)
val_split_df = val_split_df.reset_index(drop=True)

y_train_split = np.array([LABEL2ID[a] for a in train_split_df['answer']])
y_val_split = np.array([LABEL2ID[a] for a in val_split_df['answer']])


def compute_metrics_mc(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }

## Model 1 

In [27]:
def row_text(df):
    return (df['prompt'] + " " + df['A'] + " " + df['B'] + " " + df['C'] + " " + df['D'] + " " + df['E']).tolist()



tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(row_text(train_split_df)).toarray()
X_val_tfidf = tfidf_vectorizer.transform(row_text(val_split_df)).toarray()
X_test_tfidf = tfidf_vectorizer.transform(row_text(test_df)).toarray()


class MCQScratchDataset(TorchDataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(MCQScratchDataset(X_train_tfidf, y_train_split), batch_size=16, shuffle=True)
val_loader = DataLoader(MCQScratchDataset(X_val_tfidf, y_val_split), batch_size=16, shuffle=False)


class ScratchMCQSolver(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super(ScratchMCQSolver, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 5)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


model_scratch = ScratchMCQSolver(input_dim=X_train_tfidf.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=1e-3)

wandb.init(project="smart-mcq-solver", name="Model-1-Scratch-MLP")
NUM_EPOCHS_SCRATCH = 10

for epoch in range(NUM_EPOCHS_SCRATCH):
    model_scratch.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model_scratch(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (torch.argmax(outputs, dim=1) == batch_y).sum().item()
        total += batch_y.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total

    model_scratch.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            outputs = model_scratch(batch_x)
            val_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy().tolist())
            val_labels.extend(batch_y.numpy().tolist())
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average="macro")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "eval_accuracy": val_acc,
        "eval_f1": val_f1,
    })
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS_SCRATCH} - loss: {train_loss:.4f} "
          f"- train_acc: {train_acc:.4f} - val_acc: {val_acc:.4f} - val_f1: {val_f1:.4f}")

model1_metrics = {"accuracy": val_acc, "f1": val_f1}
wandb.finish()

Epoch 1/10 - loss: 1.2624 - train_acc: 0.6972 - val_acc: 0.9950 - val_f1: 0.9950
Epoch 2/10 - loss: 0.2250 - train_acc: 1.0000 - val_acc: 1.0000 - val_f1: 1.0000
Epoch 3/10 - loss: 0.0377 - train_acc: 1.0000 - val_acc: 1.0000 - val_f1: 1.0000
Epoch 4/10 - loss: 0.0153 - train_acc: 1.0000 - val_acc: 1.0000 - val_f1: 1.0000
Epoch 5/10 - loss: 0.0085 - train_acc: 1.0000 - val_acc: 1.0000 - val_f1: 1.0000
Epoch 6/10 - loss: 0.0054 - train_acc: 1.0000 - val_acc: 1.0000 - val_f1: 1.0000
Epoch 7/10 - loss: 0.0038 - train_acc: 1.0000 - val_acc: 1.0000 - val_f1: 1.0000
Epoch 8/10 - loss: 0.0028 - train_acc: 1.0000 - val_acc: 1.0000 - val_f1: 1.0000
Epoch 9/10 - loss: 0.0022 - train_acc: 1.0000 - val_acc: 1.0000 - val_f1: 1.0000
Epoch 10/10 - loss: 0.0017 - train_acc: 1.0000 - val_acc: 1.0000 - val_f1: 1.0000


epoch,▁▂▃▃▄▅▆▆▇█
eval_accuracy,▁█████████
eval_f1,▁█████████
train_accuracy,▁█████████
train_loss,█▂▁▁▁▁▁▁▁▁
epoch,10
eval_accuracy,1
eval_f1,1
train_accuracy,1
train_loss,0.00171


## Model 2 

In [28]:
def fetch_context(prompt_text):
    docs = retriever.invoke(prompt_text)
    return " ".join([doc.page_content for doc in docs])

train_split_df['context'] = train_split_df['prompt'].apply(fetch_context)
val_split_df['context'] = val_split_df['prompt'].apply(fetch_context)
test_df['context'] = test_df['prompt'].apply(fetch_context)


MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_mcq_with_rag(examples):
    first_sentences = [
        [f"Context: {ctx}\nQuestion: {prompt}"] * 5 
        for ctx, prompt in zip(examples['context'], examples['prompt'])
    ]
    
    second_sentences = [
        [f"A: {a}", f"B: {b}", f"C: {c}", f"D: {d}", f"E: {e}"]
        for a, b, c, d, e in zip(examples['A'], examples['B'], examples['C'], examples['D'], examples['E'])
    ]
    
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(first_sentences, second_sentences, truncation=True, max_length=384)
    
    return {k: [v[i:i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}


from datasets import Dataset as HFDataset

hf_train_ds = HFDataset.from_pandas(train_split_df)
encoded_train_ds = hf_train_ds.map(preprocess_mcq_with_rag, batched=True, remove_columns=hf_train_ds.column_names)
encoded_train_ds = encoded_train_ds.add_column("label", y_train_split.tolist())

hf_val_ds = HFDataset.from_pandas(val_split_df)
encoded_val_ds = hf_val_ds.map(preprocess_mcq_with_rag, batched=True, remove_columns=hf_val_ds.column_names)
encoded_val_ds = encoded_val_ds.add_column("label", y_val_split.tolist())

hf_test_ds = HFDataset.from_pandas(test_df)
encoded_test_ds = hf_test_ds.map(preprocess_mcq_with_rag, batched=True, remove_columns=hf_test_ds.column_names)


from dataclasses import dataclass
from typing import Optional, Union
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
import torch

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features] if label_name in features[0] else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flat_features = [[{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features]
        flat_features = sum(flat_features, [])

        batch = self.tokenizer.pad(
            flat_features, padding=self.padding, max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of, return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [29]:
model_2 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

training_args_m2 = TrainingArguments(
    output_dir="./deberta_mcq_results",
    eval_strategy="epoch",          
    save_strategy="epoch",
    learning_rate=1e-5,
    warmup_steps=50,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    weight_decay=0.01,
    adam_epsilon=1e-6,
    logging_steps=10,
    max_grad_norm=0.5,
    fp16=False,
    report_to="wandb",
    run_name="Model-2-DeBERTa-v3-Full",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_m2 = Trainer(
    model=model_2,
    args=training_args_m2,
    train_dataset=encoded_train_ds,
    eval_dataset=encoded_val_ds,       
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_mc, 
)

trainer_m2.train()
model2_metrics = trainer_m2.evaluate()
print("Model 2 validation metrics:", model2_metrics)
wandb.finish()

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                  

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

Model 2 validation metrics: {'eval_loss': 3.21875, 'eval_accuracy': 0.235, 'eval_f1': 0.2294631301188009, 'eval_runtime': 3.6114, 'eval_samples_per_second': 55.381, 'eval_steps_per_second': 6.923, 'epoch': 3.0}


eval/accuracy,▄▄▁█
eval/f1,▁▅▂█
eval/loss,▁▁▁▁
eval/runtime,▁▇▃█
eval/samples_per_second,█▂▆▁
eval/steps_per_second,█▂▆▁
train/epoch,▁▁▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇█████
train/global_step,▁▁▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇█████
train/grad_norm,█▃▃▃▁▁▁▁▁▁▁▁▂▁▁▁▁
train/learning_rate,▂▄▅▇██▇▆▆▅▅▄▃▃▂▂▁
+1,...


## Model 3

In [30]:
model_3_base = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["classifier", "pooler"],
)

model_3 = get_peft_model(model_3_base, lora_config)
model_3.print_trainable_parameters()

training_args_m3 = TrainingArguments(
    output_dir="./deberta_lora_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    warmup_steps=50,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    logging_steps=10,
    max_grad_norm=0.5,
    fp16=False,
    report_to="wandb",
    run_name="Model-3-DeBERTa-LoRA",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_m3 = Trainer(
    model=model_3,
    args=training_args_m3,
    train_dataset=encoded_train_ds,
    eval_dataset=encoded_val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_mc,
)

trainer_m3.train()
model3_metrics = trainer_m3.evaluate()
print("Model 3 validation metrics:", model3_metrics)
wandb.finish()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                  

trainable params: 886,273 || all params: 185,309,186 || trainable%: 0.4783


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,42.141211,5.187500,0.175000,0.176599
2,29.682715,3.238281,0.255000,0.248526
3,27.579687,3.173828,0.350000,0.347955


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Model 3 validation metrics: {'eval_loss': 3.173828125, 'eval_accuracy': 0.35, 'eval_f1': 0.34795491719484767, 'eval_runtime': 4.0251, 'eval_samples_per_second': 49.688, 'eval_steps_per_second': 6.211, 'epoch': 3.0}


eval/accuracy,▁▄██
eval/f1,▁▄██
eval/loss,█▁▁▁
eval/runtime,▆█▄▁
eval/samples_per_second,▃▁▅█
eval/steps_per_second,▄▁▅█
train/epoch,▁▁▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇█████
train/global_step,▁▁▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇█████
train/grad_norm,▁▁▁▁▁▄▅▆█▃▃▂▃▆▃▃▄
train/learning_rate,▂▄▅▇██▇▆▆▅▅▄▃▃▂▂▁
+1,...


In [34]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score

wandb.init(project="smart-mcq-solver", name="Model-Final-Ensemble-Inference")

model_scratch.eval()
with torch.no_grad():
    logits_m1_val = model_scratch(torch.tensor(X_val_tfidf, dtype=torch.float32).to(device)).cpu().numpy()
    logits_m1_test = model_scratch(torch.tensor(X_test_tfidf, dtype=torch.float32).to(device)).cpu().numpy()

logits_m2_val = trainer_m2.predict(encoded_val_ds).predictions
logits_m2_test = trainer_m2.predict(encoded_test_ds).predictions

logits_m3_val = trainer_m3.predict(encoded_val_ds).predictions
logits_m3_test = trainer_m3.predict(encoded_test_ds).predictions

f1_scores = np.array([
    model1_metrics["f1"],
    model2_metrics["eval_f1"],
    model3_metrics["eval_f1"]
])

if f1_scores.sum() <= 1e-8:
    weights = np.array([1 / 3, 1 / 3, 1 / 3])
else:
    weights = f1_scores / f1_scores.sum()

w1, w2, w3 = weights
print(f"Ensemble weights -> Model1: {w1:.3f}, Model2: {w2:.3f}, Model3: {w3:.3f}")

T = 1.2
probs_m1_val = F.softmax(torch.tensor(logits_m1_val) / T, dim=-1).numpy()
probs_m2_val = F.softmax(torch.tensor(logits_m2_val) / T, dim=-1).numpy()
probs_m3_val = F.softmax(torch.tensor(logits_m3_val) / T, dim=-1).numpy()


ensemble_probs_val = (w1 * probs_m1_val) + (w2 * probs_m2_val) + (w3 * probs_m3_val)
ensemble_val_preds = np.argmax(ensemble_probs_val, axis=1)

ensemble_val_acc = accuracy_score(y_val_split, ensemble_val_preds)
ensemble_val_f1 = f1_score(y_val_split, ensemble_val_preds, average="macro")

print(f"Ensemble Validation Accuracy: {ensemble_val_acc:.4f}, Macro-F1: {ensemble_val_f1:.4f}")
print(f"  vs Model 1 val F1: {model1_metrics['f1']:.4f}")
print(f"  vs Model 2 val F1: {model2_metrics['eval_f1']:.4f}")
print(f"  vs Model 3 val F1: {model3_metrics['eval_f1']:.4f}")

wandb.log({
    "ensemble_val_accuracy": ensemble_val_acc,
    "ensemble_val_f1": ensemble_val_f1,
    "weight_model1": w1,
    "weight_model2": w2,
    "weight_model3": w3,
})


probs_m1_test = F.softmax(torch.tensor(logits_m1_test) / T, dim=-1).numpy()
probs_m2_test = F.softmax(torch.tensor(logits_m2_test) / T, dim=-1).numpy()
probs_m3_test = F.softmax(torch.tensor(logits_m3_test) / T, dim=-1).numpy()


ensemble_probs_test = (w1 * probs_m1_test) + (w2 * probs_m2_test) + (w3 * probs_m3_test)

final_predictions = []
for row_probs in ensemble_probs_test:
    top_3_indices = np.argsort(row_probs)[-3:][::-1]
    final_predictions.append(" ".join(LABELS[i] for i in top_3_indices))

id_col = 'ID' if 'ID' in test_df.columns else 'id' if 'id' in test_df.columns else test_df.index
submission_df = pd.DataFrame({
    'ID': test_df[id_col] if id_col in test_df.columns else id_col,
    'Prediction': final_predictions,
})
submission_df.to_csv('submission.csv', index=False)

print("Ensemble Saved successfully with Softmax probability averaging!")
wandb.finish()

test/accuracy,▁█
test/f1,▁█
test/loss,█▁
test/runtime,▁▇▂█
test/samples_per_second,▆█▁▂
test/steps_per_second,▆█▁▂
test/accuracy,0.35
test/f1,0.34795
test/loss,3.17383
test/runtime,9.976
test/samples_per_second,50.121


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Ensemble weights -> Model1: 0.634, Model2: 0.145, Model3: 0.221
Ensemble Validation Accuracy: 1.0000, Macro-F1: 1.0000
  vs Model 1 val F1: 1.0000
  vs Model 2 val F1: 0.2295
  vs Model 3 val F1: 0.3480
Ensemble Saved successfully with Softmax probability averaging!


ensemble_val_accuracy,▁
ensemble_val_f1,▁
test/accuracy,▁█
test/f1,▁█
test/loss,█▁
test/runtime,▁▆▁█
test/samples_per_second,▇█▂▁
test/steps_per_second,▇█▂▁
weight_model1,▁
weight_model2,▁
+1,...
